# Intensity servo: sensor calibration, closed-loop FRA, and AOM transmission

This notebook uses the **new intensity-calibration bitstream**. The existing fast PI and both laser PLLs remain present. It measures/exports tables; it does not apply an inverse table to feedback.

**Wiring:** DAC **228/3** at your existing **80 MHz** carrier → RF amplifier → AOM. Optical output → PDA36A2 → ZAD-1+ IF; DAC **228/1** supplies the existing mixer LO (normally 90 MHz); mixer RF → ADC **224/1**. Keep the receiver's working RFDC NCO/phase, LO amplitude, detector gain and cabling fixed throughout sensor calibration and subsequent measurements. The optical power meter measures the same stabilized output (or a fixed pickoff).

The dither is added after the PI and a new **one-clock boundary register (4.069 ns)**. This fixed register is present with or without the test tone; it separates the existing PI output saturation from the diagnostic adder for timing closure. The PI algorithm and gains are unchanged. The measured return ratio includes this delay. Each DMA record contains four raw I samples, the registered PI/manual command C, the actual limited DAC command U, and the dither phase. Python converts each raw sample to optical power before averaging. No new feedback filter or lookup table is enabled.

Reprogramming interrupts all existing locks and resets their state. Attaching with `download=False` and refreshing Python do not reprogram hardware.

### Use this notebook section by section
Do **not** use Run All for the first calibration: the sensor points and optical setpoints require your measurements. Work in this order: (1) save the working setup, (2) load the matched image, (3) measure the sensor table, (4) test a small FRA point and sweep, (5) validate a quasi-static band, (6) scan/revisit setpoints, (7) review and export. To do FRA only later, reuse the sensor table for the **same unchanged sensor/LO setup** and skip the AOM scan.

### What remains available
| Existing function | This image |
|---|---|
| Intensity PI arithmetic, gains, bias, IIR, anti-windup and integrator rules | Preserved |
| Manual optical setpoint and original 32-bit streamed setpoints | Preserved; this measurement notebook uses manual setpoints |
| Manual DAC amplitude control and intensity mixer LO | Preserved |
| Both independent laser PLLs, tracking/capture controls and optional lead-lag paths | Preserved |
| Laser I/Q DMA, closed-loop FRA and RF phase-modulation FRA | Preserved |
| RFDC configuration and existing register addresses | Preserved |

The deliberate feedback-path change is the **4.069 ns intensity output delay** described above. Diagnostic command limits apply while the dither is active; at idle, the registered original command passes through. The inverse table stays on disk. Loading any bitstream resets hardware state even though its functions are retained; saving the intensity snapshot does **not** save or restore the other laser locks.


In [ ]:
from pathlib import Path
from datetime import datetime
import importlib, json, time, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import intensity_calibration as intensity
importlib.invalidate_caches()
intensity = importlib.reload(intensity)
print("Helper:", intensity.__file__)
output_dir = Path("intensity_measurements") / datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir.mkdir(parents=True, exist_ok=True)
setup_path = Path("intensity_working_setup.json")


## Before loading the new bitstream: save the working intensity settings
Run this once against your existing `ol`, while using a nonzero **manual optical setpoint**. The new helper can read the old R7 setup. The snapshot preserves mixer settings, GPIOs, PI gains and biases. It cannot preserve the PI integrator or a loaded serial setpoint sequence. If already running the new bitstream, skip saving over your original snapshot.

If `ol` is not defined in this kernel, first attach to the **currently loaded image's matching bit/HWH** with `ol = Overlay("path/to/current.bit", download=False)` after importing `Overlay` from `pynq`. Do not initialize clocks to take this snapshot. Use your normal controls to select a stable manual optical setpoint before saving; the notebook does not reconstruct an active streamed sequence. Keep a copy of `intensity_working_setup.json` alongside your experiment notes.


In [ ]:
save_working_setup = False  # Change to True once, BEFORE reprogramming.
if save_working_setup:
    working = intensity.configuration(ol)
    assert working["gpio"]["11"][0] & 1, "Select your working manual setpoint first."
    assert working["gpio"]["11"][1] & 0xFFFF, "Zero setpoint disables the PI."
    setup_path.write_text(json.dumps(working, indent=2))
    print("Saved:", setup_path.resolve())


## Load or attach to the matching overlay
Copy `TransportPhaseLockFF.bit` and `.hwh` from the new package together. For a first download, preserve the working setup above, set `download_hardware=True`, and run this cell. For a new notebook on an already-running new image, leave it False. The clock initialization below is the existing external-reference configuration you supplied.

Copy `intensity_calibration.py` and this notebook into the same Jupyter folder (or ensure the printed helper path is the intended file). Edit `bitfile` to the new package's actual location. First download requires stopping/restarting your experiment as usual; the restore cell restores only the saved intensity registers and the three listed mixer settings, not all RFDC runtime settings or integrator state. Check the optical operating point before continuing. For an already-running image, use `download_hardware=False`; this branch does not change clocks or reset loops. The expected intensity signature is **0x494E5401**; the laser ABI remains **0xA7**, which alone cannot distinguish this image from an older R7 bitstream.


In [ ]:
from pynq import Overlay
import xrfdc, xrfclk
bitfile = Path("/home/xilinx/Bitfiles/FreqLocksBranch/overlays/TransportPhaseLockFF.bit")
download_hardware = False
if download_hardware:
    assert setup_path.exists(), "Save your working intensity setup before reprogramming."
    custom_lmk_file = Path("/home/xilinx/jupyter_notebooks/LMK_245_76_ExtRef10.txt")
    custom_regs = []
    for line in custom_lmk_file.read_text().splitlines():
        match = re.search(r"(0x[0-9A-F]+)", line, re.IGNORECASE)
        if match:
            custom_regs.append(int(match.group(1), 16))
    assert custom_regs, "No LMK registers found."
    if not xrfclk.xrfclk.lmk_devices or not xrfclk.xrfclk.lmx_devices:
        xrfclk.xrfclk._find_devices()
        xrfclk.xrfclk._read_tics_output()
    lmk_chip = xrfclk.xrfclk.lmk_devices[0]["compatible"]
    xrfclk.xrfclk._Config[lmk_chip][245.761] = custom_regs
    xrfclk.set_ref_clks(lmk_freq=245.761, lmx_freq=491.52)
    time.sleep(2)
    ol = Overlay(str(bitfile))
    rfdc = ol.usp_rf_data_converter_0
    time.sleep(2)
    rfdc.adc_tiles[0].StartUp()
    rfdc.dac_tiles[0].StartUp()
    time.sleep(1)
    intensity.restore_manual_configuration(ol, json.loads(setup_path.read_text()))
elif "ol" not in globals():
    ol = Overlay(str(bitfile), download=False)
rfdc = ol.usp_rf_data_converter_0
intensity.check(ol)
print("Intensity ABI:", hex(ol.intensity_limits.read(8)))
print("Working settings:", intensity.configuration(ol))
# Restore any other experiment/PLL branches using their normal bring-up cells.


## Sensor calibration: manual setpoint, measured optical power
First take a dark/electronic-offset record with the optical input blocked, without changing detector gain or mixer settings; set its optical power to zero after subtracting any unwanted additive optical background. `capture()` does not change the output. Then unblock and take points across the operating range using your usual manual setpoint. Avoid ADC/mixer compression and unstable/nonmonotonic branches.

A zero manual setpoint resets/disables the existing PI; with its closed-loop output selected, `intensity.manual_setpoint(ol, 0)` commands zero RF amplitude. For a dark record, disable the intensity output using that or your normal controls before blocking the detector, so the PI does not rail against the blocked light. Restore a known working nonzero setpoint before illuminated measurements. Include enough calibration coverage beyond subsequent dither excursions and noise; the analysis refuses extrapolation. The calibration uses raw ADC counts, before the PI's `adc_bias` addition.

Run the initialization cell below once, then rerun the **next** cell once per measured point. At each illuminated point, select a known stable manual setpoint, wait for optical power to settle, read the power meter, then enter that value when prompted. Include at least three distinct points, and use more closely spaced points where curvature is apparent. Check that power is still stable during the short ADC capture. The plotted counts-to-power curve must be single-valued and monotonic; the helper accepts either increasing or decreasing polarity.

The setpoint is in **signed ADC counts**, not milliwatts and not DAC codes; use your existing sign/bias convention. Actual AOM drive is reported separately as `u0_codes`, normally within 0…32767. Avoid demanding power that pins the controller at either limit. Near-full-scale calibration needs headroom for the dither rather than operating exactly at a rail. A fixed power-meter pickoff ratio only changes the normalization; its ratio must remain fixed.

### Reusing measurements taken on the previous bitstream
The PI and setpoint mapping are unchanged, so setpoint-versus-power measurements made before this build can be reused with unchanged sensor/LO/ADC/bias settings. The existing PI forms `error = floor(mean(raw_I)) + adc_bias - setpoint`. For illuminated points with settled, unsaturated integral feedback and negligible residual error, `raw_adc_counts ≈ signed_setpoint_counts - signed_adc_bias_counts` (up to rounding/residual error). Save the bias with those measurements; preferably record actual raw counts when available. Do **not** use this inference for a zero-setpoint dark point: zero disables the PI, so obtain the dark raw-count offset from an actual capture. Convert prior illuminated points to the same raw-count convention before constructing `sensor_table`; do not mix setpoint codes and raw ADC counts in one table.


In [ ]:
sensor_points = []  # Run once; do not rerun this cell between points.
sensor_csv = output_dir / "sensor_calibration_points.csv"
power_units = "mW"  # Or "relative"; use consistently throughout.


In [ ]:
# Run once per manually measured power point, including a dark point.
# Optional setpoint change (omit for the dark record or while keeping current power):
# intensity.manual_setpoint(ol, 5000)  # ADC counts; choose a known working value.
# time.sleep(0.5)
measured_power = float(input(f"Offset-corrected optical power [{power_units}]: "))
record = intensity.capture(ol, samples=32768, decimation_log2=8)
i_counts, c_codes, u_codes, phase = intensity.unpack(record)
assert not record["status"] & ((1 << 2) | (1 << 3) | (1 << 6) | (1 << 9)), "Invalid/railed sensor capture"
filename = output_dir / f"sensor_point_{len(sensor_points):03d}.npz"
np.savez_compressed(filename, **record,
                    metadata_json=json.dumps(intensity.configuration(ol)))
sensor_points.append(dict(adc_counts=float(i_counts.mean()),
                          adc_std_counts=float(i_counts.std()),
                          optical_power=measured_power, power_units=power_units,
                          u0_codes=float(u_codes.mean()), raw_file=str(filename)))
pd.DataFrame(sensor_points).to_csv(sensor_csv, index=False)
display(pd.DataFrame(sensor_points))


In [ ]:
points = pd.read_csv(sensor_csv)
sensor = intensity.sensor_table(points.adc_counts, points.optical_power, power_units=power_units)
(output_dir / "sensor_table.json").write_text(json.dumps(sensor, indent=2))
plt.plot(sensor["adc_counts"], sensor["optical_power"], "o-")
plt.xlabel("Raw ADC I [counts]")
plt.ylabel(f"Offset-corrected optical power [{power_units}]")
plt.grid(); plt.show()


## First FRA point and amplitude check
Keep feedback active at a fixed nonzero manual setpoint. Start with a small command dither; **32 codes is an illustrative start**, not a guaranteed 1% optical modulation. Check the measured `fractional_modulation`, actual command `U`, clipping and repeat scatter. Aim initially for roughly 0.5–2% optical modulation if tolerable. Repeat at half amplitude and at twice the sample rate. Large U cancellation means you need another frequency, more clean blocks, or a deliberately lower calibration-loop bandwidth.

The software fits offset + linear drift + cosine + sine; phasors use `xc - i*xs`. `L=-C/U` is the full closed-loop return ratio including the unchanged raw-sensor PI. `K=Q/(P0*U)` is the fractional optical plant response, including the RF driver and sensor dynamics. They are different quantities.

A single point is a quick commissioning check, **not** a precision result: `flags` should be empty, the optical operating point should remain stable, and the dither should be small. Then use repeated blocks. `fractional_modulation` is a **peak fraction**, so 0.01 means 1% peak (2% peak-to-peak). DAC `amplitude_codes` is also peak. Increasing it is not a remedy for hardware clipping or a sensor table with insufficient range. At very high loop gain, C nearly cancels the injection and U can become too small to resolve; an enormous value of -C/U in that situation is not a reliable loop-gain measurement.


In [ ]:
point = intensity.measure_point(
    ol, 1000, amplitude_codes=32, table=sensor,
    minimum_measure_s=0.02, raw_path=output_dir / "first_point.npz")
print({key: point[key] for key in
       ("frequency_hz", "u0_codes", "p0", "fractional_modulation",
        "power_drift_fraction", "flags", "loop", "K")})


## Closed-loop FRA, 10 Hz to 1 MHz
The settings below average repeated short records. Raw records are point samples, without a new temporal antialias filter: increasing capture rate and checking convergence is necessary if appreciable noise/harmonics lie above its Nyquist frequency. Longer averaging reduces random error but does not remove alias bias, clipping, calibration errors or upstream modulation coherent with the dither.

The acquisition planner uses >=16 samples/cycle and approximately integer-cycle blocks. It retains every raw block and excludes blocks with clipping, invalid data, controller rails, >3% fitted power drift, or >2% fitted command drift. The quality test uses repeated complex-response scatter. Use a separate, stronger low-frequency sweep if needed; avoid large probes around crossover.

Read the plots as follows: the left panels show the dimensionless loop return ratio L; a reliable 0 dB crossing is a candidate crossover. Phase is displayed modulo 360 degrees, so choose the physically continuous branch before inferring margin. The right panels show K in inverse DAC codes; a flat, nearly real region is a candidate static calibration band. `loop_reliable` and `plant_reliable` are separate: one can be usable while the other is not. Points failing those criteria remain in the CSV but are omitted from the plotted curves.

At each frequency the helper alternates dither polarity, averages accepted complex responses, and retries up to `max_repeats`. Rejected blocks remain saved with a reason. If too many blocks drift, shorten them while retaining enough cycles; if U is unresolved, try a modestly larger probe only where the loop stays linear, or change the calibration frequency/loop settings deliberately. Do not interpret a sweep through visible oscillation or rails as a small-signal FRA.

The default sweep stores about **0.86 GiB before compression** at eight blocks per point (about 2.57 GiB at 24); allow board disk space. The acquisition/settling floor is about two minutes, with Python fitting, compression and retries adding time. Start with a shorter frequency list if commissioning. Each rerun creates a fresh file; previously recorded blocks are not overwritten.


In [ ]:
frequencies_hz = np.geomspace(10, 1e6, 61)
fra_csv = output_dir / f"intensity_loop_{datetime.now():%H%M%S_%f}.csv"
rows = intensity.sweep(
    ol, frequencies_hz, amplitude_codes=32, table=sensor,
    repeats=8, max_repeats=24, target_snr=10,
    maximum_drift_fraction=0.03, minimum_measure_s=0.02,
    csv_path=fra_csv)
print("Saved:", fra_csv.resolve())


In [ ]:
df = pd.read_csv(fra_csv)
loop_good = df.loop_reliable.astype(str).str.lower().eq("true")
plant_good = df.plant_reliable.astype(str).str.lower().eq("true")
f_loop = df.loc[loop_good, "frequency_hz"].to_numpy()
f_plant = df.loc[plant_good, "frequency_hz"].to_numpy()
L = (df.loc[loop_good, "loop_re"] + 1j*df.loc[loop_good, "loop_im"]).to_numpy()
K = (df.loc[plant_good, "K_re"] + 1j*df.loc[plant_good, "K_im"]).to_numpy()
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
axes[0, 0].semilogx(f_loop, 20*np.log10(abs(L)), "o-")
axes[1, 0].semilogx(f_loop, np.angle(L, deg=True), "o-")
axes[0, 1].loglog(f_plant, abs(K), "o-")
axes[1, 1].semilogx(f_plant, np.angle(K, deg=True), "o-")
axes[0, 0].set_ylabel("Return ratio |L| [dB]")
axes[0, 1].set_ylabel("Fractional plant |K| [1/DAC code]")
for ax in axes[1]: ax.set_xlabel("Modulation frequency [Hz]"); ax.set_ylabel("Phase [degrees]")
for ax in axes.flat: ax.grid(True, which="both", alpha=.3)
fig.tight_layout(); fig.savefig(output_dir / "intensity_fra.png", dpi=180)
plt.show()


## Check the proposed quasi-static calibration band
Select the frequency after reviewing the FRA. At one fixed operating point, compare nearby frequencies, half amplitude, a shorter block, and twice the capture rate. These measurements should agree within their repeated-block uncertainty. The response phase must be close to 0 or 180 degrees before treating its real part as a static slope. Do not correct phase merely to force agreement.


In [ ]:
calibration_frequency_hz = 1000  # Illustrative: select from the measured plateau.
calibration_amplitude_codes = 32  # Adjust for ~0.5–2% optical modulation.
base_D = intensity.plan(calibration_frequency_hz)["decimation_log2"]
assert base_D > 3, "Choose a slower quasi-static calibration band for the doubled-rate check."
checks = [
    ("nominal", calibration_frequency_hz, calibration_amplitude_codes, {}),
    ("half_amplitude", calibration_frequency_hz, max(1, calibration_amplitude_codes//2), {}),
    ("lower_frequency", 0.8*calibration_frequency_hz, calibration_amplitude_codes, {}),
    ("higher_frequency", 1.2*calibration_frequency_hz, calibration_amplitude_codes, {}),
    ("shorter_block", calibration_frequency_hz, calibration_amplitude_codes,
     {"minimum_measure_s": 0.01, "cycles": 8}),
    ("double_sample_rate", calibration_frequency_hz, calibration_amplitude_codes,
     {"decimation_log2": base_D-1}),
]
check_rows = []
check_stamp = datetime.now().strftime("%H%M%S_%f")
for label, frequency_hz, amplitude, window in checks:
    result = intensity.sweep(
        ol, [frequency_hz], amplitude_codes=amplitude, table=sensor,
        repeats=8, max_repeats=24, csv_path=output_dir / f"check_{check_stamp}_{label}.csv", **window)[0]
    check_rows.append(dict(check=label, frequency_hz=result["frequency_hz"],
                           K_real=result["K"].real, K_imag=result["K"].imag,
                           phase_deg=np.angle(result["K"], deg=True),
                           K_sem=result["K_complex_sem"], reliable=result["plant_reliable"]))
check_table = pd.DataFrame(check_rows)
check_table.to_csv(output_dir / f"static_band_checks_{check_stamp}.csv", index=False)
display(check_table)


## AOM nonlinearity: scan operating points with feedback active
Choose a dither frequency from the **measured nearly real, flat portion of K**, above substantial upstream drift and preferably above a deliberately reduced calibration-loop bandwidth. The 1 kHz value below is only a starting example; detector gain, AOM beam size/acoustic transit and RF-driver response determine the usable plateau.

Enter manual setpoints spanning the useful command range, then revisit them in reverse order. Re-running the scan cell appends uniquely named files to `calibration_files`, allowing later revisits to be combined; use a new output directory for a different sensor calibration or RF setup. Each result is associated with measured `u0_codes`, not the requested setpoint. The amplitude, frequency, half-amplitude and block-duration checks should agree before integrating slopes. Large drift within a block, or changing thermal state at a repeated command, invalidates the simple multiplicative model.

Start with a short list of nearby, known stable setpoints before attempting the full range. Inspect the actual mean command and its excursions at each point. The code scans forward and backward and leaves the final setpoint selected; it does not automatically restore the initial operating point. Add later revisits only while sensor calibration, detector gain, RF settings and optical geometry are unchanged. Incident power may drift between visits—that is the effect the normalized method is intended to cancel—but a changed actuator curve cannot be normalized away.


In [ ]:
setpoints_counts = []  # Fill with known stable, nonzero manual ADC-count setpoints.
# Uses the frequency/amplitude checked in the preceding cell.
if "calibration_files" not in globals():
    calibration_files = []
scan_stamp = datetime.now().strftime("%H%M%S_%f")
assert setpoints_counts, "Enter the manual setpoints to measure."
for index, setpoint in enumerate(setpoints_counts + setpoints_counts[::-1]):
    intensity.manual_setpoint(ol, int(setpoint))
    time.sleep(0.5)  # Increase if the optical power/thermal state has not settled.
    filename = output_dir / f"aom_slope_{scan_stamp}_{index:03d}.csv"
    intensity.sweep(
        ol, [calibration_frequency_hz], amplitude_codes=calibration_amplitude_codes,
        table=sensor, repeats=8, max_repeats=24, minimum_measure_s=0.02,
        maximum_drift_fraction=0.03, csv_path=filename)
    calibration_files.append(filename)
# The final optical setpoint remains selected; use your normal manual setting to restore it.


## Review slopes, bin by actual command, reconstruct and export
First compare nearby dither frequencies, half amplitude, shorter duration and revisits under different incident power. A phase near 0 or 180 degrees is necessary but not sufficient for a static slope. `bin_slopes` preserves signed real K; it never substitutes |K|. Bins use empirical scatter, without hiding disagreements in a smoothing fit. Unresolved command or optical modulation is rejected using within-block fit estimates; also inspect the repeated FRA quality flags.

F is recovered only up to normalization. This export contains an inverse on one monotonic branch, with no extrapolation. Applying it to the live PI, preserving the operating point during enable, and independently validating dP/P≈dv/v are future steps; this bitstream intentionally does not implement live compensation.

The default command-bin width is 256 DAC codes and requires at least three accepted, resolved blocks per bin. Reduce the bin width if the curve changes appreciably within it; obtain more measurements rather than forcing sparse bins to pass. The reconstruction rejects gaps greater than 1024 codes by default. Check the slope-versus-command plot and revisit scatter before setting `quasi_static_and_revisits_verified=True`. If k changes sign, restrict the data to one monotonic branch before inversion. The exported transmission is normalized to the maximum on the measured branch; it is not an absolute diffraction efficiency.

### Saved files and resuming analysis
Everything goes under the `output_dir` printed/defined near the top, relative to the notebook's working directory:

- `sensor_calibration_points.csv`, raw sensor-point NPZs and `sensor_table.json`: measured sensor calibration, units and raw records.
- Each sweep's `.csv`: averaged complex C/U/Q, L/K, uncertainties and quality flags. `_blocks.csv`: every fitted block and rejection reason. Matching `.json`: settings and sensor table. Matching subdirectory: raw NPZ per attempt.
- `aom_signed_slopes.csv`, `aom_relative_transmission.csv`, `aom_inverse_lut.csv`: reviewed slopes and exported relative inverse, with floating and rounded DAC codes.

To revisit an existing dataset in a new kernel, set `output_dir = Path("intensity_measurements/YOUR_SAVED_DIRECTORY")`, load `sensor = json.loads((output_dir / "sensor_table.json").read_text())`, and set `calibration_files` to the aggregate `aom_slope_*.csv` paths **excluding** files ending `_blocks.csv`. Analysis/export cells work from disk and do not change hardware. Use the same sensor table only while its physical settings remain applicable. The detailed units, raw 16-byte frame layout and register map are in `docs/intensity_calibration.md` in the package.


In [ ]:
blocks = pd.concat([pd.read_csv(p.with_name(p.stem + "_blocks.csv"))
                    for p in calibration_files], ignore_index=True)
blocks["accepted"] = blocks.accepted.astype(str).str.lower().eq("true")
binned = pd.DataFrame(intensity.bin_slopes(
    blocks.to_dict("records"), bin_width_codes=256, maximum_phase_deg=10))
assert len(binned) >= 3, "Need more resolved command bins on a quasi-static branch."
binned.to_csv(output_dir / "aom_signed_slopes.csv", index=False)
plt.errorbar(binned.u_codes, binned.k_per_code, yerr=binned.k_sem, fmt="o")
plt.xlabel("Actual mean command u [DAC codes]")
plt.ylabel("Signed d ln(F) / du [1/DAC code]")
plt.grid(); plt.show()


In [ ]:
quasi_static_and_revisits_verified = False  # Set True only after reviewing those checks.
assert quasi_static_and_revisits_verified
inverse = intensity.reconstruct(
    binned.u_codes, binned.k_per_code, maximum_gap_codes=1024)
pd.DataFrame({"u_codes": inverse["u_codes"], "F_relative": inverse["F_relative"]}).to_csv(
    output_dir / "aom_relative_transmission.csv", index=False)
v = np.linspace(inverse["inverse_transmission"][0], inverse["inverse_transmission"][-1], 1024)
u = intensity.inverse_command(inverse, v)
pd.DataFrame({"relative_transmission": v, "u_codes_float": u,
              "u_codes_rounded": np.rint(u).astype(int)}).to_csv(
    output_dir / "aom_inverse_lut.csv", index=False)
plt.plot(inverse["u_codes"], inverse["F_relative"], "o-")
plt.xlabel("Actual command [DAC codes]"); plt.ylabel("Relative transmission F")
plt.grid(); plt.show()
print("Export directory:", output_dir.resolve())


## Stop a diagnostic
This ramps off the intensity dither only. It leaves the PI, manual setpoint, LO and RF carrier settings untouched. A failed capture also removes the dither in `finally`; integrity faults are not accepted as usable transfer data.

Interrupting an active Python measurement normally executes its `finally` cleanup and ramps the tone off. After an interruption, run the stop cell and check the optical power. If the kernel was killed, attach to the already loaded matching image without downloading, reload the helper and run `intensity.stop(ol)`. This does not relock or reset the PI. Return any deliberately changed setpoint or loop settings using your usual controls.


In [ ]:
intensity.stop(ol)
